In [30]:
from dataset_loader import load_arxpr_data

from llama_index.core import Document
from llama_index.core.node_parser import get_leaf_nodes, get_root_nodes
from llama_index.core.node_parser import (SentenceSplitter, MarkdownNodeParser, MarkdownElementNodeParser)

from llama_index.core import VectorStoreIndex
from llama_index.core import Settings
from llama_index.core.schema import TextNode
from llama_index.llms.ollama import Ollama
from llama_index.embeddings.ollama import OllamaEmbedding
from llama_index.core.ingestion import IngestionPipeline

from llama_index.core.extractors import (
    # SummaryExtractor,
    QuestionsAnsweredExtractor,
    KeywordExtractor,
    # BaseExtractor,
)


import nest_asyncio
nest_asyncio.apply()



## Markdown Parser

In [31]:

def _pseudo_markdown_splitter(text: str, chunk_size=2000, chunk_overlap=125, markdown_headers=[]):
    """
    Borrowing the LangChain markdown splitter to spot header str in lieu of #, ##, ###, etc. 
    Keeps the md-formated header in medatdata dict if found, else nothing.
    Then recursively splits the text without breaking paragraphs.
    
    Args:
        text: str
            The text from UnstructuredXMLLoader in line-separated format header, subheader, pargaraphs.
        ...
        headers_to_split_on: list
            A list of tuples of the form (header, metadata_key) where header is a str that
            will be used to split the text and metadata_key is the key in the metadata dict
            that will be used to store ONLY markdown-formatted header.

    Returns:
        splits: list
            A list of Langchain doc objects, each containing paragraphs and artifact sub-headers according to 
            chunk size; and 'metadata' key would contain real markdown headers IF any.
    
    """

    from langchain_text_splitters import MarkdownHeaderTextSplitter
    from langchain_text_splitters import RecursiveCharacterTextSplitter

    # split by headers
    markdown_splitter = MarkdownHeaderTextSplitter(markdown_headers)
    md_header_splits = markdown_splitter.split_text(text)

    # chunk
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=chunk_size, chunk_overlap=chunk_overlap)
    splits = text_splitter.split_documents(md_header_splits)
    
    return splits


In [32]:

def _match_sequence(seq1: str, seq2_list: list, threshold=0.7):
    """
    Gives a sequence similarity match between seq1 and seq2 from seq2_list.

    Args: 
        seq1: str
            The sequence to match
        seq_list: list
            A list of sequences to match against seq1
        threshold: float
            The minimum similarity ratio; <.8 for short sequences seems to work well.
    Return:
        seq2: str
            The sequence that passes the matching threshold or None if insufficient match.
            
    """

    from difflib import SequenceMatcher

    for seq2 in seq2_list:
        if SequenceMatcher(None, seq1, seq2).ratio() > threshold:
            return seq2
    return None
        


def _split_non_md_headers(doc_lc, headers=[]):
    """ A secondery cleaner to extract leftover non-markdown headers from the text. 
    
    Args:
        doc_lc: LangChain Document 
            A list of Langchain doc objects, each containing paragraphs and artifact sub-headers according to 
            chunk size; and 'metadata' key would contain real markdown headers IF any.
        headers: list    
            A list of possible headers (e.g. headers = ["introduction", "paragraph", "title_1", "title_2", "fig_caption"])
    Return:
        cleaned_docs: dict
            Contains core paragraphs under dict['text'] and associated headers under dict['headers']

    """

    cleaned_docs = {}
    cleaned_docs['text'] = ""
    cleaned_docs['headers'] = []
    text = doc_lc.page_content

    for line in text.split("\n"):
        match = _match_sequence(line.lower(), headers)
        if match:
            cleaned_docs['headers'].append(match)
        else:
            cleaned_docs['text'] += line
    return cleaned_docs

In [ ]:
t,l = load_arxpr_data(5)

# keys for manual check on papers
paper_keys = ['25918225', '18618715', '28845460', '25977295', '18631455']
text = t['18631455']

markdown_headers = [
    ("METHODS", "methods"),
    ("METHODOLOGY", "methods"),   
    ("RESULT", "result"),
    ("RESULTS", "result"),
    ("FIG", "figure"),
    ("FIGURE", "figure"),
    ("INTRO", "introduction"),
    ("INTRODUCTION", "introduction"),
    ("REF", "reference"),
    ("REFERENCES", "reference"),
    ("DISCUSS", "discussion"),
    ("DISCUSSION", "discussion"),
    ("SUPPL", "supplement"),
    ("SUPPLEMENT", "supplement"),
    ("abstract_title_1", "abstract"),
]

# TODO: need a script to generate a list of headers from a large set or sampled papers.
other_headers = ["introduction",
                "paragraph",
                "title_1",
                "title_2",
                "fig_caption",
                "abstract",
                "supplementary material",
                "materials and methods",
                "results",
                "discussion",
                "results and discussion",
                "footnote_title",
                "figures and tables",
                "summary", 
                "ref",
                "lancetref",
                "experimental procedures"
                ]
                

Settings.embed_model = OllamaEmbedding(model_name="llama3")
Settings.llm = Ollama(model="llama3") 

metadata_extractors = [
    # QuestionsAnsweredExtractor(questions=3),
    # KeywordExtractor(keywords=5)
    ]

pipeline = IngestionPipeline(transformations=metadata_extractors)

nodes = []
splits = _pseudo_markdown_splitter(text, chunk_size=2000, chunk_overlap=125, markdown_headers=markdown_headers)

for doc in splits:
    split = _split_non_md_headers(doc, headers=other_headers)

    print(f'\nMD Headers: {doc.metadata} \nOther Headers: {split["headers"]} \nSize change: {len(doc.page_content)} ? {len(split["text"])}')
    print(f'{split["text"]}')

    # Avoid creating a node with empty text
    if len(split["text"]) == 0: continue

    nodes.append(TextNode(text=split['text']))

docs = pipeline.run(documents=nodes)
index = VectorStoreIndex(nodes=docs)


N datasets with exactly one label, for each field:
{'assay_by_molecule_14': 3,
 'assay_count_7': 4,
 'experimental_design_10': 0,
 'experimental_factors_20': 2,
 'hardware_4': 1,
 'name_19': 4,
 'no_of_samples_22': 0,
 'no_of_samples_23': 0,
 'organism_16': 3,
 'releasedate_12': 4,
 'sample_count_13': 4,
 'sex_2': 0,
 'study_type_18': 3,
 'technology_15': 4,
 'type_21': 0,
 'type_9': 3}
N datasets with at least one label, for each field
{'assay_by_molecule_14': 5,
 'assay_count_7': 4,
 'experimental_design_10': 0,
 'experimental_factors_20': 4,
 'hardware_4': 1,
 'name_19': 5,
 'no_of_samples_22': 0,
 'no_of_samples_23': 0,
 'organism_16': 5,
 'releasedate_12': 5,
 'sample_count_13': 5,
 'sex_2': 1,
 'study_type_18': 5,
 'technology_15': 5,
 'type_21': 5,
 'type_9': 5}

MD Headers: {} 
Other Headers: ['title_1', 'abstract'] 
Size change: 1317 ? 1267
BioC-APIcollection.keyauthor_manuscriptImmunityThis file is available for text mining. It may also be used consistent with the principles 

In [5]:
from llama_index.core.response.notebook_utils import display_source_node

base_retriever = index.as_retriever(similarity_top_k=3)

query_str = "What is the use of RT-PCR kit?"
base_nodes = base_retriever.retrieve(query_str)


In [7]:
for node in base_nodes: 
    print('')
    print(node.get_content(metadata_mode='all'))
    # print(node.metadata)
    # print(node.embedding)
    # display_source_node(node, source_length=500)


[Excerpt from document]
excerpt_keywords: Here are five unique keywords for this document:

Real-time PCR, Drosophila, RNAi, Quantitative RT-PCR, BuGZ
Excerpt:
-----
Quantitative real-time PCR  Total RNA was isolated from Drosophila larvae or pupae using an RNA isolation kit (RNeasy Plus Mini kit; QIAGEN) and the manufacturer's protocol. Quantitative RT-PCR was performed using the iScript One Step RT-PCR kit (170-8892; Bio-Rad Laboratories) on a real-time PCR detection system (CFX96; Bio-Rad Laboratories). 50 ng of total RNA was reverse transcribed and amplified as follows: 50 C for 10 min, 95 C for 5 min, 95 C for 10 s, 60 C for 30 s, 72 C for 1 min. Steps 2-4 were repeated for 40 cycles. Each reaction was performed in triplicate, and the results of three independent experiments were used for statistical analysis. Relative mRNA expression levels were quantified using the DeltaDeltaC(t) method. Results were normalized to those for RP49, and primer sequences were as follows. Rp49 (Dros

## Semantic Parsing

In [ ]:
from llama_index.core.node_parser import SemanticSplitterNodeParser
from llama_index.embeddings.openai import OpenAIEmbedding

embed_model = OpenAIEmbedding()
splitter = SemanticSplitterNodeParser(
    buffer_size=1, breakpoint_percentile_threshold=95, embed_model=embed_model
)
